# Olist Warehouse Analysis

This notebook connects to the modeled DuckDB warehouse and produces the metrics needed for the executive presentation.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WAREHOUSE_PATH = PROJECT_ROOT / 'data' / 'warehouse' / 'module2_project.duckdb'

engine = create_engine(f'duckdb:///{WAREHOUSE_PATH}')
sns.set_theme(style='whitegrid')

## Warehouse Inventory

In [ ]:
with engine.connect() as conn:
    tables = pd.read_sql(
        text("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema IN ('raw', 'warehouse')
        ORDER BY table_schema, table_name
        """),
        conn,
    )

tables

In [ ]:
with engine.connect() as conn:
    row_counts = pd.read_sql(
        text("""
        SELECT 'dim_customer' AS table_name, COUNT(*) AS row_count FROM warehouse.dim_customer
        UNION ALL SELECT 'dim_product', COUNT(*) FROM warehouse.dim_product
        UNION ALL SELECT 'dim_seller', COUNT(*) FROM warehouse.dim_seller
        UNION ALL SELECT 'dim_date', COUNT(*) FROM warehouse.dim_date
        UNION ALL SELECT 'fact_sales', COUNT(*) FROM warehouse.fact_sales
        ORDER BY table_name
        """),
        conn,
    )

row_counts

## Executive KPIs

Transaction values are GMV-style marketplace activity measures, not confirmed Olist revenue.

In [ ]:
with engine.connect() as conn:
    kpis = pd.read_sql(
        text("""
        SELECT
            COUNT(DISTINCT order_id) AS orders,
            COUNT(*) AS order_items,
            COUNT(DISTINCT seller_key) AS active_sellers,
            ROUND(SUM(price), 2) AS product_gmv,
            ROUND(SUM(total_sale_amount), 2) AS gross_transaction_value,
            ROUND(SUM(price) / COUNT(DISTINCT seller_key), 2) AS product_gmv_per_seller,
            ROUND(SUM(total_sale_amount) / COUNT(DISTINCT order_id), 2) AS average_order_value,
            ROUND(AVG(review_score), 2) AS average_review_score,
            ROUND(AVG(delivery_days), 1) AS average_delivery_days
        FROM warehouse.fact_sales
        WHERE order_status = 'delivered'
        """),
        conn,
    )

kpis

## Monthly GMV Trend

In [ ]:
with engine.connect() as conn:
    monthly_gmv = pd.read_sql(
        text("""
        SELECT
            date_trunc('month', d.full_date)::DATE AS month_start,
            SUM(fs.price) AS product_gmv,
            SUM(fs.total_sale_amount) AS gross_transaction_value,
            COUNT(DISTINCT fs.order_id) AS orders,
            COUNT(DISTINCT fs.seller_key) AS active_sellers
        FROM warehouse.fact_sales AS fs
        INNER JOIN warehouse.dim_date AS d
            ON fs.order_date_key = d.date_key
        WHERE fs.order_status = 'delivered'
        GROUP BY month_start
        ORDER BY month_start
        """),
        conn,
    )

monthly_gmv.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=monthly_gmv, x='month_start', y='product_gmv', marker='o', ax=ax)
ax.set_title('Monthly Delivered Product GMV')
ax.set_xlabel('Month')
ax.set_ylabel('Product GMV')
plt.xticks(rotation=45)
plt.tight_layout()

## Seller Health

Olist's primary clients are sellers, so this section evaluates seller productivity, concentration, and retention proxies.

In [ ]:
with engine.connect() as conn:
    seller_concentration = pd.read_sql(
        text("""
        WITH seller_gmv AS (
            SELECT
                seller_key,
                SUM(price) AS product_gmv
            FROM warehouse.fact_sales
            WHERE order_status = 'delivered'
            GROUP BY seller_key
        ),
        ranked AS (
            SELECT
                seller_key,
                product_gmv,
                ROW_NUMBER() OVER (ORDER BY product_gmv DESC) AS seller_rank,
                SUM(product_gmv) OVER () AS total_gmv
            FROM seller_gmv
        )
        SELECT
            ROUND(SUM(CASE WHEN seller_rank <= 10 THEN product_gmv ELSE 0 END) / MAX(total_gmv), 4) AS top_10_share,
            ROUND(SUM(CASE WHEN seller_rank <= 100 THEN product_gmv ELSE 0 END) / MAX(total_gmv), 4) AS top_100_share,
            ROUND(SUM(CASE WHEN seller_rank <= 500 THEN product_gmv ELSE 0 END) / MAX(total_gmv), 4) AS top_500_share
        FROM ranked
        """),
        conn,
    )

seller_concentration

In [ ]:
with engine.connect() as conn:
    seller_states = pd.read_sql(
        text("""
        SELECT
            ds.seller_state,
            COUNT(DISTINCT fs.seller_key) AS active_sellers,
            COUNT(DISTINCT fs.order_id) AS orders,
            COUNT(*) AS items,
            ROUND(SUM(fs.price), 2) AS product_gmv,
            ROUND(SUM(fs.total_sale_amount), 2) AS gross_transaction_value,
            ROUND(AVG(fs.review_score), 2) AS average_review_score,
            ROUND(AVG(fs.delivery_days), 2) AS average_delivery_days
        FROM warehouse.fact_sales AS fs
        INNER JOIN warehouse.dim_seller AS ds
            ON fs.seller_key = ds.seller_key
        WHERE fs.order_status = 'delivered'
        GROUP BY ds.seller_state
        ORDER BY product_gmv DESC
        LIMIT 10
        """),
        conn,
    )

seller_states

## Product Category Performance

In [ ]:
with engine.connect() as conn:
    category_gmv = pd.read_sql(
        text("""
        SELECT
            COALESCE(dp.product_category_name_english, 'unknown') AS category,
            ROUND(SUM(fs.price), 2) AS product_gmv,
            ROUND(SUM(fs.total_sale_amount), 2) AS gross_transaction_value,
            COUNT(DISTINCT fs.order_id) AS orders,
            ROUND(AVG(fs.review_score), 2) AS average_review_score
        FROM warehouse.fact_sales AS fs
        LEFT JOIN warehouse.dim_product AS dp
            ON fs.product_key = dp.product_key
        WHERE fs.order_status = 'delivered'
        GROUP BY category
        ORDER BY product_gmv DESC
        LIMIT 10
        """),
        conn,
    )

category_gmv

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=category_gmv, y='category', x='product_gmv', ax=ax)
ax.set_title('Top Product Categories By Product GMV')
ax.set_xlabel('Product GMV')
ax.set_ylabel('Category')
plt.tight_layout()

## Customer Segmentation

In [ ]:
with engine.connect() as conn:
    customer_segments = pd.read_sql(
        text("""
        WITH customer_orders AS (
            SELECT
                dc.customer_unique_id,
                dc.customer_state,
                COUNT(DISTINCT fs.order_id) AS orders,
                SUM(fs.price) AS product_gmv
            FROM warehouse.fact_sales AS fs
            INNER JOIN warehouse.dim_customer AS dc
                ON fs.customer_key = dc.customer_key
            WHERE fs.order_status = 'delivered'
            GROUP BY dc.customer_unique_id, dc.customer_state
        )
        SELECT
            CASE
                WHEN orders >= 3 THEN 'repeat 3+'
                WHEN orders = 2 THEN 'repeat 2'
                ELSE 'one-time'
            END AS segment,
            COUNT(*) AS customers,
            ROUND(SUM(product_gmv), 2) AS product_gmv,
            ROUND(AVG(product_gmv), 2) AS average_customer_value
        FROM customer_orders
        GROUP BY segment
        ORDER BY product_gmv DESC
        """),
        conn,
    )

customer_segments

## Recommendation Notes

- Use seller GMV concentration to identify key accounts that need proactive support.
- Compare high-GMV sellers and categories with review scores to find quality or fulfillment risks.
- Use customer geography to prioritize regional demand opportunities and seller coverage.
- Use delivery variance to identify fulfillment improvements that may protect seller retention and customer satisfaction.
- Treat GMV as platform activity, not Olist revenue, unless take-rate or internal finance data is added.